# Math Knowledge Graph Crawler — Fast Version

Three speedups over v1:
1. **Batch classification** — check up to 50 pages' categories per HTTP call using the raw MediaWiki API (instead of 1 call per page via `wikipedia-api`).
2. **Parallel requests** — use a thread pool so multiple batches are in flight at once.
3. **Disk cache** — save `is_math_cache` and the graph to disk so crashes don't lose progress.

Expected runtime: ~5–10 min for 2000 nodes at depth 3, vs. ~30–60 min before.

In [ ]:
%pip install wikipedia-api networkx matplotlib pyvis tqdm requests

In [ ]:
import requests
import wikipediaapi
import networkx as nx
import time
import json
import re
import pickle
import os
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

USER_AGENT = "MathKnowledgeGraphResearch/0.2 (catherine.cho.27@dartmouth.edu)"
API_URL = "https://en.wikipedia.org/w/api.php"

# Shared session with connection pooling — reuses TCP connections across calls
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

# Still used for fetching summaries/URL of nodes we KEEP (small number of calls)
wiki = wikipediaapi.Wikipedia(
    user_agent=USER_AGENT,
    language="en",
    extract_format=wikipediaapi.ExtractFormat.WIKI,
)

print("Setup complete.")

## Filter config (same as before)

In [ ]:
MATH_KEYWORDS = {
    "mathematic", "algebra", "geometry", "calculus", "topology",
    "analysis", "number theory", "combinatoric", "probability",
    "statistic", "logic", "set theory", "graph theory", "linear algebra",
    "differential equation", "theorem", "functional analysis",
    "measure theory", "category theory", "discrete mathematic",
    "numerical analysis", "optimization", "cryptograph",
}

EXCLUDE_KEYWORDS = {
    "births", "deaths", "people from", "alumni", "universities",
    "living people", "20th-century", "21st-century", "19th-century",
}

BAD_NAMESPACES = (
    "Portal:", "Category:", "Template:", "Template talk:",
    "File:", "Wikipedia:", "Help:", "Draft:", "Module:",
    "MediaWiki:", "User:", "Talk:", "Book:",
)

YEAR_RE = re.compile(r"^\d{3,4}s?$|^\d+(st|nd|rd|th) century$")

BAD_EXACT = {
    "ISBN", "ISSN", "Doi (identifier)", "PMID (identifier)",
    "Bibcode (identifier)", "Zbl (identifier)", "MR (identifier)",
    "ArXiv (identifier)", "OCLC (identifier)",
    "Glossary of mathematical symbols",
    "List of mathematical symbols",
}

def is_obviously_not_concept(title: str) -> bool:
    if title.startswith(BAD_NAMESPACES):
        return True
    if YEAR_RE.match(title):
        return True
    if title in BAD_EXACT:
        return True
    if "(identifier)" in title:
        return True
    if title.startswith("List of films"):
        return True
    return False

def classify_from_categories(cats: list[str]) -> bool:
    """Given a list of category names, decide if page is a math concept."""
    if not cats:
        return False
    cats_lower = [c.lower() for c in cats]
    math_hits = sum(any(kw in c for kw in MATH_KEYWORDS) for c in cats_lower)
    if any(any(ex in c for ex in EXCLUDE_KEYWORDS) for c in cats_lower):
        return math_hits >= 2
    return math_hits >= 1

## Batched category fetcher

Uses the raw MediaWiki API's `prop=categories` with up to 50 titles per request.

One quirk: if a page has many categories, the API paginates with a `continue` token. We handle that inside `fetch_categories_batch`.

In [ ]:
def fetch_categories_batch(titles: list[str]) -> dict[str, list[str]]:
    """Return {title: [category_name, ...]} for up to 50 titles in one API call."""
    assert len(titles) <= 50
    result = {t: [] for t in titles}

    params = {
        "action": "query",
        "format": "json",
        "prop": "categories",
        "titles": "|".join(titles),
        "cllimit": "max",     # up to 500 categories per page per response
        "clshow": "!hidden",  # skip maintenance categories
    }

    while True:
        r = session.get(API_URL, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()

        pages = data.get("query", {}).get("pages", {})
        # Also handle normalization (e.g., "linear Algebra" -> "Linear algebra")
        norm_map = {n["from"]: n["to"] for n in data.get("query", {}).get("normalized", [])}

        for _, pg in pages.items():
            title = pg.get("title")
            # Find the original requested title for this page
            original = next((orig for orig, norm in norm_map.items() if norm == title), title)
            if original not in result:
                original = title  # fallback
            if original in result:
                for c in pg.get("categories", []):
                    cat_name = c["title"].replace("Category:", "")
                    result[original].append(cat_name)

        if "continue" in data:
            params.update(data["continue"])
        else:
            break

    return result


def classify_many(titles: list[str], max_workers: int = 8) -> dict[str, bool]:
    """Classify many titles in parallel batches of 50."""
    # Split into chunks of 50
    chunks = [titles[i:i+50] for i in range(0, len(titles), 50)]
    results: dict[str, bool] = {}

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        for cats_dict in pool.map(fetch_categories_batch, chunks):
            for title, cats in cats_dict.items():
                results[title] = classify_from_categories(cats)

    return results

## Batched link fetcher

Same idea for fetching the outgoing links from a page — do it in batches.

In [ ]:
def fetch_links_batch(titles: list[str]) -> dict[str, list[str]]:
    """Return {title: [link_title, ...]} for up to 50 titles. Main-namespace links only."""
    assert len(titles) <= 50
    result = {t: [] for t in titles}

    params = {
        "action": "query",
        "format": "json",
        "prop": "links",
        "titles": "|".join(titles),
        "pllimit": "max",
        "plnamespace": "0",  # 0 = main namespace (real articles, no Portal:/Category:/etc.)
    }

    while True:
        r = session.get(API_URL, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()

        pages = data.get("query", {}).get("pages", {})
        norm_map = {n["from"]: n["to"] for n in data.get("query", {}).get("normalized", [])}

        for _, pg in pages.items():
            title = pg.get("title")
            original = next((orig for orig, norm in norm_map.items() if norm == title), title)
            if original not in result:
                original = title
            if original in result:
                for l in pg.get("links", []):
                    result[original].append(l["title"])

        if "continue" in data:
            params.update(data["continue"])
        else:
            break

    return result


def fetch_links_many(titles: list[str], max_workers: int = 8) -> dict[str, list[str]]:
    chunks = [titles[i:i+50] for i in range(0, len(titles), 50)]
    results: dict[str, list[str]] = {}
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        for d in pool.map(fetch_links_batch, chunks):
            results.update(d)
    return results

## Disk cache

Persists `is_math_cache` between runs. If you crash or interrupt the crawl, just rerun — already-classified pages are skipped.

In [ ]:
CACHE_FILE = "math_cache.pkl"

def load_cache() -> dict[str, bool]:
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "rb") as f:
            return pickle.load(f)
    return {}

def save_cache(cache: dict[str, bool]) -> None:
    with open(CACHE_FILE, "wb") as f:
        pickle.dump(cache, f)

## Main crawler — BFS in layers

Key structural change: instead of popping one page at a time, we process a **whole BFS layer at once**. This lets us batch every API call and parallelize. Also much faster to reason about depth.

In [ ]:
def build_math_graph_fast(
    seeds: list[str],
    max_depth: int = 3,
    max_nodes: int = 2000,
    max_workers: int = 8,
    save_every_layer: bool = True,
):
    G = nx.DiGraph()
    is_math_cache = load_cache()
    print(f"Loaded {len(is_math_cache)} cached classifications")

    current_layer = set(seeds)

    for depth in range(max_depth + 1):
        if not current_layer or len(G.nodes) >= max_nodes:
            break

        print(f"\n=== Layer {depth}: {len(current_layer)} candidates ===")

        # 1. Filter syntactic junk
        candidates = [t for t in current_layer if not is_obviously_not_concept(t)]

        # 2. Classify anything we haven't seen
        unknown = [t for t in candidates if t not in is_math_cache]
        if unknown:
            print(f"  Classifying {len(unknown)} new titles...")
            new_classifications = classify_many(unknown, max_workers=max_workers)
            is_math_cache.update(new_classifications)
            if save_every_layer:
                save_cache(is_math_cache)

        # 3. Keep only the math ones
        math_titles = [t for t in candidates if is_math_cache.get(t, False)]
        print(f"  {len(math_titles)} classified as math")

        # 4. Add them as nodes (respecting cap)
        for t in math_titles:
            if t not in G and len(G.nodes) < max_nodes:
                G.add_node(t)

        if depth == max_depth:
            break  # don't fetch links on final layer

        # 5. Fetch links for all math pages in this layer (batched & parallel)
        to_fetch = [t for t in math_titles if t in G]
        print(f"  Fetching links for {len(to_fetch)} pages...")
        links_map = fetch_links_many(to_fetch, max_workers=max_workers)

        # 6. Build the next layer
        next_layer = set()
        for src, targets in links_map.items():
            for tgt in targets:
                if is_obviously_not_concept(tgt):
                    continue
                next_layer.add(tgt)
                # Add edge tentatively; we'll prune non-math targets after classification
                G.add_edge(src, tgt)

        current_layer = next_layer
        print(f"  Graph so far: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    # Final cleanup: remove any edge pointing to a non-math node, and drop orphan targets
    print("\nCleaning up non-math nodes...")
    to_remove = [n for n in G.nodes if not is_math_cache.get(n, False)]
    G.remove_nodes_from(to_remove)
    save_cache(is_math_cache)
    print(f"Final: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    return G

## Run it

In [ ]:
SEEDS = ["Algebra", "Calculus", "Linear algebra", "Topology", "Probability theory"]

G = build_math_graph_fast(
    SEEDS,
    max_depth=3,
    max_nodes=2000,
    max_workers=8,
)

print(f"\nDone. Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

In [ ]:
# Quick sanity check
top_in = sorted(G.in_degree, key=lambda x: x[1], reverse=True)[:20]
print("Most linked-to topics:")
for title, deg in top_in:
    print(f"  {deg:4d}  {title}")

In [ ]:
# Save outputs
nx.write_graphml(G, "math_graph.graphml")
with open("math_graph.json", "w") as f:
    json.dump(nx.node_link_data(G), f, indent=2)

from pyvis.network import Network
net = Network(height="750px", width="100%", directed=True, notebook=False)
net.from_nx(G)
net.force_atlas_2based()
net.save_graph("math_graph.html")
print("Saved: math_graph.graphml, math_graph.json, math_graph.html")